# Class 5 (benchmark version): We Have the Data. Let's Settle It.

### 🧭 The one question we are answering

In **Class 1** you trained a classifier on 10,000 patients. In **Class 2** you fitted a line
to ten house sales. Both took labelled data, a training step, and an honest evaluation.

In the gentle version of this class we asked a language model to do those jobs *without* any
training data, and eyeballed how it went. Today we stop eyeballing.

> **We put the language model on exactly the same benchmark as the models you trained —
> the same patients, the same houses, the same metrics — and read the scoreboard.**

No cherry-picked demo, no vibes. Accuracy, precision, recall, F1, AUC, average error, cost
per prediction. The numbers you already know how to read.

### 🗺️ Roadmap

| Part | What we do | Time |
|---|---|---|
| 0 | Warm-up: sentiment on customer reviews — where the model looks brilliant | 15 min |
| 1 | **Class 1's benchmark**: 10,000 patients, five approaches, one scoreboard | 35 min |
| 2 | **Class 2's benchmark**: ten houses, tested honestly for the first time | 20 min |
| 3 | Cost, speed, and the decision you will actually have to make | 15 min |

### ⚠️ Two rules we are keeping from Classes 1 and 2

1. **Never judge a model on data it was trained on.** Everything below is measured on
   held-out data, for every approach, including the language model.
2. **A number means nothing without a baseline.** The dumbest possible model is on every
   chart, and it is not always where you expect it.

### 🐍 About the Python

Every cell can be run with **Shift + Enter**, top to bottom. The tasks marked ✏️ ask you to
change *one sentence in quotes* — the prompt. That sentence is the model, so it is the only
thing worth editing.

In [1]:
# If running on Google Colab, clone the repo (if needed),
# move into the repo directory, and ensure it’s on the Python path.

import sys, os

def in_colab():
    try: import google.colab; return True
    except: return False

if in_colab():
    repo = "Hands-On-Notebooks"
    if os.path.basename(os.getcwd()) != repo:
        if not os.path.exists(repo):
            !git clone https://github.com/BridgingAISocietySummerSchools/{repo}
        %cd {repo}
    if '.' not in sys.path:
        sys.path.append('.')

In [2]:
# Setup — run this once. Nothing here calls the API yet.

import time
import numpy as np

# The sentiment warm-up, borrowed unchanged from the gentle version of this class.
from plotting_utils.llm_simple import (
    REVIEWS,                    # 8 customer reviews with agreed-on labels
    HOUSE_SIZES, HOUSE_PRICES,  # the same 10 houses as Class 2
    to_label, to_number, to_price_in_thousands,
    show_label_results, plot_accuracy_bars,
)

# Class 1's own plots, so the language model is judged with the same instruments.
from plotting_utils.classification import plot_confusion_matrix, plot_roc_curves

# The benchmark machinery for today.
from plotting_utils.llm_benchmark import (
    run_in_parallel,            # several API calls at once, so 50 of them take seconds
    load_screening_benchmark,   # Class 1's data, split and model, plus a benchmark sample
    patient_to_text,            # one row of the patient table -> one sentence
    labelled_examples,          # solved training patients, as text for the prompt
    score_classification, show_scoreboard, plot_metric_comparison,
    leave_one_out_line, other_houses_as_text,
    score_regression, plot_price_comparison, plot_mae_bars,
    cost_projection, plot_cost_scaleup,
)

print("✅ Ready.")

✅ Ready.


### 🔑 Connect to a Real Model

Everything today talks to a **real language model** through
[OpenRouter](https://openrouter.ai/). **Your instructor will give you a key for the session.**

> ⚠️ **Never type an API key into a notebook cell.** Notebooks get saved, committed,
> screenshotted and shared — and a key in a notebook is a key on the internet.

The cell below asks for the key in a **hidden input box**, so it stays in memory and never
lands in the saved file. If a key is already in a `.env` file or in Colab's 🔑 **Secrets**
panel (add `OPENROUTER_API_KEY` there, and switch **Notebook access** on), it is picked up
automatically and you will not be prompted.

The loading logic lives in [`llm_client.py`](llm_client.py) — short, and worth a look.

In [3]:
import os, getpass
from llm_client import ask_llm, llm_available, describe_setup, print_usage, USAGE

if not llm_available():
    try:
        key = getpass.getpass("Paste the workshop key (hidden): ").strip()
    except Exception:                 # no interactive input available
        key = ""
    if key:
        os.environ["OPENROUTER_API_KEY"] = key

# A benchmark means volume. This notebook makes ~130 calls, so we deliberately
# pick a **small, fast** model rather than the biggest one available: it answers
# in well under a second and costs a small fraction of a frontier model.
# (Delete this line to use whatever OPENROUTER_MODEL your .env sets instead.)
os.environ["OPENROUTER_MODEL"] = "anthropic/claude-haiku-4.5"

LLM_READY = describe_setup()

🔌 Connected. Model: anthropic/claude-haiku-4.5
   Key loaded from a .env file or the environment — never from this notebook.


### ⚡ Which model, how much data, how many calls

Unlike the earlier notebooks, this one runs a **benchmark**, and a benchmark means volume.
With the settings below it makes roughly **100 API calls** — which is the point: you are about
to find out what that costs, and Part 3 puts the number on screen.

**Where the time goes, and what we sample.** Class 1's model is trained on all 10,000 of its
patients, because fitting a logistic regression to 7,000 rows takes about **three
milliseconds**. Nothing local is worth economising on. What costs time and money is the
language model, at a second and a fraction of a cent per patient — so *that* is what we
sample: a few dozen held-out patients rather than all 3,000. Class 2's data needs no sampling
at all; it is ten houses, and we use every one.

That sample size is not free either, and Part 1 shows you exactly what it costs you in
certainty. Turn `N_PATIENTS` down to 20 for a faster run-through, or up to 100 if you want the
error bars to shrink and have four minutes to spare.

That is also why the cell above pins a **small, fast model** (`claude-haiku-4.5`) instead of
the largest one on offer. Three reasons, and all three are the real reasons people do this in
production:

- **Speed.** Under a second per answer instead of several. Across 130 calls that is the
  difference between a coffee break and a workshop.
- **Cost.** Roughly an order of magnitude cheaper per token. At benchmark volumes that shows
  up on the bill immediately.
- **It is usually enough.** "Read this sentence and give me one number" is not a task that
  needs a frontier model — and *finding out* which size is enough for your job is exactly the
  kind of question a benchmark like this one exists to answer.

> 🔬 **Worth trying at the end:** change that one line to a bigger model and re-run Part 1.
> If the scoreboard barely moves, you have just saved your project a lot of money.

The one knob worth touching is `N_PATIENTS`. Fifty patients is already a small benchmark
(Part 1 shows you *how* small). Turn it down to 20 if you are in a hurry or paying for it
yourself; turn it up to 100 if you want the confidence intervals to shrink.

In [4]:
# ⏱️ Teaching-speed settings. These are the only numbers worth touching.

N_PATIENTS = 30      # held-out patients we send to the model  (30 → 60 API calls)
N_EXAMPLES = 20      # solved patients pasted into the prompt in Step 4
TASK_PATIENTS = 12   # a smaller sample again, for the ✏️ task in Part 1

## Part 0: The Warm-Up — Where the Model Looks Brilliant ⏱️ 15 min

Before the hard test, the easy one. This is the example from the gentle version of this
class, and it is here for a reason: **it is the case the language model wins outright**, and
you should see that clearly before you see it lose.

One function does everything: **`ask`**. Text in, text out. That is the entire interface.

In [5]:
def ask(question):
    """Send a question to the model and return its answer as text."""
    return ask_llm(question, max_tokens=200)


print(ask("In two sentences, what is machine learning?"))

Machine learning is a type of artificial intelligence where computers learn patterns from data and improve their performance on tasks without being explicitly programmed for each scenario. It works by identifying patterns in training data and using those patterns to make predictions or decisions on new, unseen data.


### 🏷️ Sentiment, With No Training Data At All

Class 1's recipe for a classifier: collect thousands of labelled examples, split them, train,
evaluate. Here is the alternative, in one sentence of English.

Two details in the prompt are doing real work:

- **`Answer with the category name only.`** Without it the model writes a paragraph, and a
  paragraph is not a label. Telling it the *shape* of the answer is most of prompting.
- **`to_label(...)`** works out which category the reply named, so `"Clearly positive."` and
  `"positive"` both count. A trained model hands you a number; a language model hands you
  prose, and turning prose back into data is work you will always have to do.

In [6]:
def classify(text, labels):
    """Ask the model to sort `text` into exactly one of `labels`."""
    prompt = (f"Classify the text into exactly one of these categories: {', '.join(labels)}.\n"
              f"Answer with the category name only.\n\n"
              f"Text: {text}")
    reply = ask_llm(prompt, max_tokens=20)
    return to_label(reply, labels)


print(classify("The battery lasts all day and the screen is gorgeous.", ["positive", "negative"]))

positive


### 📊 Now Grade It — Against Something Embarrassing

Eight reviews, each with a label we agreed on in advance. The model never sees the labels;
they exist only so we can grade it, exactly like Class 1's test set.

And because Class 1 drilled it into us, the model does not get to compete against nothing.
Its opponents:

- **"Always say positive"** — no reading required.
- **Keyword counting** — count nice words, count nasty words, take the bigger pile. Ten
  lines, no AI, runs in a microsecond, costs nothing.

> 📌 Note that we still need labelled data here. Not to *train* the model — to **trust** it.

In [7]:
# The dumbest possible baselines, for scale. Always check these first.

GOOD = ["great", "gorgeous", "excellent", "flawlessly", "best", "worth", "nice", "wonderful"]
BAD = ["broken", "cheap", "stopped", "crashes", "slow", "never"]

def keyword_rule(text):
    """No AI at all: count nice words vs nasty words."""
    t = text.lower()
    return "positive" if sum(w in t for w in GOOD) > sum(w in t for w in BAD) else "negative"


truth = [r["label"] for r in REVIEWS]
scores = {
    "Always say 'positive'": sum(t == "positive" for t in truth) / len(truth),
    "Keyword counting": sum(keyword_rule(r["text"]) == t for r, t in zip(REVIEWS, truth)) / len(truth),
}

if LLM_READY:
    guesses = run_in_parallel(lambda r: classify(r["text"], ["positive", "negative"]), REVIEWS)
    scores["Language model"] = show_label_results(
        [r["text"] for r in REVIEWS], truth, guesses,
    )

plot_accuracy_bars(scores, title="Sentiment on the same 8 reviews")

········  (8 calls)


Review,Our label,Model said,
The battery lasts all day and the screen is gorgeous. Best …,positive,positive,✅
"Arrived broken, and support never replied to any of my thre…",negative,negative,✅
Does exactly what it promises. No complaints at all.,positive,positive,✅
Cheap plastic. It stopped working after two weeks.,negative,negative,✅
Setup took ten minutes and it has worked flawlessly ever si…,positive,positive,✅
"Oh it's wonderful, if you enjoy reading a 60-page manual to…",negative,negative,✅
"The sound quality is excellent, but the app crashes every s…",negative,negative,✅
"Shipping was painfully slow, but honestly the product is wo…",positive,positive,✅



🎯 The model agreed with us on 8 of 8 reviews (100%) — with zero training examples.


### 💬 Read That Before Moving On

Look at *which* reviews it got right. The last three are the interesting ones:

| Review | Why it's hard |
|---|---|
| *"if you enjoy reading a 60-page manual…"* | **Sarcasm.** Every individual word is positive. |
| *"sound quality is excellent, but the app crashes"* | **Good then bad.** Which half is the verdict? |
| *"shipping was slow, but worth the wait"* | **Bad then good.** Same problem, opposite order. |

Keyword counting has no chance on those, and no amount of extra keywords would fix it.
The language model reads them the way you do — because understanding sentences is what it
was built for. **On free text, with zero labelled examples, it is genuinely excellent.**

Hold on to that impression. Now we hand it a spreadsheet.

## Part 1: Class 1's Benchmark — 10,000 Patients ⏱️ 35 min

### 🏥 The setup, unchanged

This is **the actual data set from Class 1**: the same synthetic screening study, the same
seed, the same 70/30 stratified split, the same logistic regression. Nothing has been
softened to make the comparison flattering in either direction.

What is new is the last line of the cell below: a small sample of **held-out** patients that
we can afford to send to a language model one at a time.

> ⚖️ **One deliberate change, and you need to know about it.** That sample is drawn
> **balanced** — half with the disease, half without — instead of at the real 10% prevalence.
> At 10%, a few dozen patients contain three or four real cases, and recall measured on three
> cases is not a measurement, it is a rumour. The price: **accuracy on this set is not comparable to
> Class 1's 92.6%.** Here, "always say healthy" scores 50%, not 90%. Changing how you sample
> your test set changes every number that comes out of it — which is itself one of the most
> useful things in this notebook.

In [8]:
bench = load_screening_benchmark(n_benchmark=N_PATIENTS)
bench.describe()

📋 Class 1's study: 10,000 patients, 10.0% of them with the disease.
   Trained on 7,000, held out 3,000.
   Class 1's model on the full held-out set: accuracy 92.6%, AUC 0.887

🎯 Our benchmark: 30 of those held-out patients (15 with the disease, 15 without).
   Deliberately balanced, so 'always say healthy' scores 50% here, not 90%.
   Every approach in this notebook is judged on these same 30 people.
   Because of that, Class 1's model also gets its threshold moved from 0.50 to 0.10 — see Step 5.


/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning:

divide by zero encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning:

overflow encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning:

invalid value encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/christophweisser/Desktop/Coding/Hands-On-Notebooks/.venv/lib/python3.11/site-pac

In [9]:
# The first few patients we are about to hand over, exactly as Class 1 stored them.
display(bench.patients.head(5))

,age,bmi,family_history,smoker,marker_a,marker_b,site,has_disease
0,61,31.4,0,0,3.22,1.02,C,0
1,43,29.6,1,1,3.89,0.82,A,1
2,60,24.4,0,0,2.86,0.99,B,0
3,26,28.6,1,0,2.22,1.00,A,0
4,77,25.4,1,0,3.74,0.58,C,0


### ✍️ Step 1: A Table Row Is Not a Question

Class 1's classifier takes six numbers. A language model takes English. So somebody has to
decide how to *say* `marker_a = 4.12` out loud — and the moment you do that, **the wording
becomes part of the model**. Rename a column, and your accuracy changes.

That is a genuinely new failure surface. There is no equivalent step in Class 1.

In [10]:
one_patient = bench.patients.iloc[0]

print(patient_to_text(one_patient))
print()
print("Truth (which the model never sees):", "DISEASE" if one_patient.has_disease else "HEALTHY")

Age: 61 years. BMI: 31.4. Close relative has had the disease: no. Current smoker: no. Blood marker A: 3.22. Blood marker B: 1.02.

Truth (which the model never sees): HEALTHY


### 🎚️ Step 2: Ask for a Risk Score, Not a Verdict

We could ask *"disease or healthy?"* — but Class 1 Part 6 taught us not to. The trained model
does not output a verdict either; it outputs a **probability**, and then somebody compares it
to a threshold. That threshold is a choice, and throwing it away throws away the ROC curve,
the AUC, and the ability to trade false alarms against missed cases.

So we ask the language model for the same thing: **a number from 0 to 100**. Then we apply
the same 0.5 threshold to both, and both get a real ROC curve.

In [11]:
def patient_risk(patient):
    """Ask the model for the probability, in percent, that this patient has the disease."""
    prompt = (
        "You are a screening tool for a research study. This is synthetic teaching data, "
        "not a real patient.\n"
        "Estimate the probability that this patient has the disease.\n"
        "Answer with a single number from 0 to 100 and nothing else.\n\n"
        f"{patient_to_text(patient)}"
    )
    reply = ask_llm(prompt, max_tokens=10)
    percent = to_number(reply)
    return None if percent is None else min(max(percent, 0.0), 100.0) / 100


if LLM_READY:
    print("Raw reply for the patient above:", patient_risk(one_patient))

Raw reply for the patient above: 0.42


### 📞 Step 3: One Call Per Patient

The next cell makes **one API call per patient**. Sending eight at a time turns a minute of
waiting into a few seconds — that is all `run_in_parallel` does, and it is the first thing
anybody building on these models reaches for. Each `·` is a finished call; an `✗` is one that
came back unusable even after a retry.

In [12]:
zero_shot_risk = None

if not LLM_READY:
    print("⏭️  This cell needs an API key — see the setup cell near the top.")
else:
    zero_shot_risk = run_in_parallel(patient_risk, bench.patients.itertuples(), workers=8)
    zero_shot_pred = [None if r is None else int(r >= 0.5) for r in zero_shot_risk]

    print("\nThe first ten risk scores it gave:",
          [None if r is None else round(r, 2) for r in zero_shot_risk[:10]])

······························  (30 calls)

The first ten risk scores it gave: [0.42, 0.72, 0.23, 0.42, 0.72, 0.42, 0.42, 0.42, 0.72, 0.42]


### 💬 Look at Those Numbers Before You Look at the Score

You will almost certainly see **the same few values repeated** — a handful of numbers doing
duty for fifty different patients, often round ones like 0.10, 0.25 or 0.70. Class 1's model
produced things like 0.0317 and 0.6042, because it was *computing* a probability from six
measurements. This one is picking a plausible-sounding number from a small set of familiar
ones, because it is predicting **the text of an answer**, not calculating a quantity.

That matters for more than tidiness. If the model only ever emits five distinct scores, its
ROC curve has five corners, its AUC has a ceiling it cannot climb past, and there is no
threshold dial to turn — the resolution simply is not there. Class 1's model gave you a
continuum; this gives you five buckets.

Keep that in mind when you see its AUC in a moment.

### 📚 Step 4: Now Give It the Training Data — In the Prompt

So far the comparison has been unfair in the language model's favour in one way (it needed no
training data) and unfair against it in another: **Class 1's model got to see 7,000 labelled
patients, and this one saw none.**

We cannot train the language model. But we can paste labelled examples into the question and
let it work them out on the spot. That trick has a name — **in-context learning** — and it is
the single most useful thing to know about these models in practice.

`N_EXAMPLES` solved patients, drawn from the same **training** set Class 1's model learned
from. Same source, same information, fed in a completely different way.

In [13]:
EXAMPLES = labelled_examples(bench, n=N_EXAMPLES)

print(f"{N_EXAMPLES} solved patients, of which the first three look like this:\n")
print("\n".join(EXAMPLES.splitlines()[:3]))
print(f"\n(That block is ~{len(EXAMPLES.split()):,} words, and it is re-sent with every single "
      f"question. Remember that in Part 3, when we look at the bill.)")

20 solved patients, of which the first three look like this:

Age: 66 years. BMI: 29.5. Close relative has had the disease: no. Current smoker: no. Blood marker A: 4.96. Blood marker B: 0.67. -> DISEASE
Age: 37 years. BMI: 25.0. Close relative has had the disease: yes. Current smoker: no. Blood marker A: 4.24. Blood marker B: 1.82. -> HEALTHY
Age: 70 years. BMI: 26.6. Close relative has had the disease: no. Current smoker: no. Blood marker A: 3.58. Blood marker B: 1.06. -> HEALTHY

(That block is ~500 words, and it is re-sent with every single question. Remember that in Part 3, when we look at the bill.)


In [14]:
def patient_risk_with_examples(patient):
    """Same question, but with solved cases from the training set pasted in first."""
    prompt = (
        "Here are patients from a screening study, each with the correct answer. "
        "This is synthetic teaching data, not real patients.\n\n"
        f"{EXAMPLES}\n\n"
        "Using the patterns in those examples, estimate the probability that this new "
        "patient has the disease.\n"
        "Answer with a single number from 0 to 100 and nothing else.\n\n"
        f"{patient_to_text(patient)}"
    )
    reply = ask_llm(prompt, max_tokens=10)
    percent = to_number(reply)
    return None if percent is None else min(max(percent, 0.0), 100.0) / 100


few_shot_risk = None

if not LLM_READY:
    print("⏭️  This cell needs an API key — see the setup cell near the top.")
else:
    few_shot_risk = run_in_parallel(patient_risk_with_examples, bench.patients.itertuples(),
                                    workers=8)
    few_shot_pred = [None if r is None else int(r >= 0.5) for r in few_shot_risk]

······························  (30 calls)


### 🎚️ Step 5: One Correction Before We Compare — Move the Threshold

There is a trap in this benchmark, and we are going to walk into it deliberately.

Class 1's model was trained where **10%** of patients are ill. We are asking it about a set
where **50%** are. The model has not got worse — the *question changed underneath it* — but at
the default 0.5 threshold it will now say "healthy" far too often and look terrible.

That is exactly the dial from Class 1 Part 6, and here is the one line of reasoning that sets
it: shifting the base rate from 10% to 50% multiplies the odds by nine, and undoing that shift
means flagging any patient whose risk clears **0.10** instead of 0.50.

So the trained model appears **twice** below: once at the naive threshold, once corrected. The
gap between those two rows is not a modelling result. It is a reminder that a benchmark can
make a perfectly good model look bad if you build it carelessly — and that most benchmarks you
will read were built by someone in a hurry.

> 📌 The language model gets no such correction, because it has no threshold to correct: it
> was never told the base rate at all. **`AUC` is the column that sidesteps this whole
> argument** — it ignores thresholds completely and asks only whether the sick patients were
> ranked above the healthy ones. If you read one number on the table, read that one.

### 🏁 The Scoreboard

Five approaches, the same patients, the metrics from Class 1. The **95% confidence
interval** on accuracy is in there too, and it is the most important column on the table.

In [15]:
results = {
    "Always say 'healthy'": score_classification(bench.truth, bench.baseline_pred),
    "Class 1's model, threshold 0.50": score_classification(
        bench.truth, bench.model_pred, risk=bench.model_risk),
    f"Class 1's model, threshold {bench.tuned_threshold:.2f}": score_classification(
        bench.truth, bench.model_pred_tuned, risk=bench.model_risk),
}
if LLM_READY:
    results["LLM, no examples"] = score_classification(
        bench.truth, zero_shot_pred, risk=zero_shot_risk)
    results[f"LLM, {N_EXAMPLES} examples"] = score_classification(
        bench.truth, few_shot_pred, risk=few_shot_risk)

table = show_scoreboard(results, title=f"Screening, on the same {N_PATIENTS} held-out patients")
plot_metric_comparison(results)

🏁 Screening, on the same 30 held-out patients



Approach,Accuracy,95% CI,Precision,Recall,F1,Caught,Missed,False alarms,AUC
Always say 'healthy',50%,33%–67%,0%,0%,0.00,0,15,0,—
"Class 1's model, threshold 0.50",63%,46%–78%,100%,27%,0.42,4,11,0,0.916
"Class 1's model, threshold 0.10",87%,70%–95%,87%,87%,0.87,13,2,2,0.916
"LLM, no examples",57%,39%–73%,62%,33%,0.43,5,10,3,0.700
"LLM, 20 examples",60%,42%–75%,57%,80%,0.67,12,3,9,0.609


### 🔲 The Four Boxes, For the Two That Matter

Accuracy collapses four numbers into one. Class 1 spent a whole section on that, so let us
not repeat the mistake: here are the actual mistakes, for the trained model and for the
best-scoring language model.

In [16]:
plot_confusion_matrix(bench.truth, bench.model_pred_tuned,
                      title=f"Class 1's model, threshold {bench.tuned_threshold:.2f} "
                            f"({N_PATIENTS} patients)")

if LLM_READY:
    best_llm = "few examples" if few_shot_risk else "no examples"
    plot_confusion_matrix(bench.truth, [0 if p is None else p for p in few_shot_pred],
                          title=f"Language model with {N_EXAMPLES} examples ({N_PATIENTS} patients)")

### 📈 And the Comparison That Ignores the Threshold Entirely

AUC asks: *if I pick one sick patient and one healthy patient at random, how often does the
model give the sick one the higher risk score?* 0.5 is a coin flip. It does not care where you
put the threshold, which makes it the fairest single number here.

In [17]:
# AUC does not care where the threshold sits, so Class 1's model appears once.
curves = {"Class 1: logistic regression": (bench.truth, bench.model_risk)}
if LLM_READY:
    curves["LLM, no examples"] = (bench.truth, [0.5 if r is None else r for r in zero_shot_risk])
    curves[f"LLM, {N_EXAMPLES} examples"] = (bench.truth, [0.5 if r is None else r for r in few_shot_risk])

aucs = plot_roc_curves(curves, title=f"Ranking {N_PATIENTS} patients by risk")

### 💬 What Just Happened — and Why

Read **your** numbers rather than trusting a story, but here is what almost always shows up,
and the reason for it.

**1. It is not guessing, and that is not luck.**
Look at its **AUC**, not its accuracy — accuracy on thirty patients is mostly noise. The AUC
is normally somewhere well above 0.5, meaning it really does rank sicker patients higher. It
has never seen this data set, but it knows what any doctor knows about `age`, `bmi`, `smoker`
and `family_history`, because those relationships are in every medical text ever written and
this data set was built to be medically plausible. **That is world knowledge standing in for
training data** — and when your columns are things the world has written about, it is a
genuinely powerful shortcut.

**2. It is still beaten by a model you could fit on a laptop in one second.**
Look at `marker_a` and `marker_b`. They are invented. Nowhere on the internet does it say
what `marker_a = 4.12` means, or that this study's marker only bites in older patients — and
that interaction is the single strongest signal in the data. Class 1's model learned it from
7,000 examples. The language model cannot know it, so it quietly ignores the most
informative column and leans on the ones it recognises.

> 📌 **This is the normal case, not a trick.** Your company's tables are full of
> `customer_score_v3`, `region_code_7`, `days_since_last_touch_norm`. World knowledge is
> worth nothing there. Whoever has the labelled history wins.

**3. Did the examples help? Check — do not assume.**
Compare the two language-model rows, and compare them on **AUC**, not accuracy. A very common
pattern is that examples make the model flag more people: recall jumps, precision falls,
accuracy barely moves, and the *ranking* — the thing AUC measures — gets no better or gets
worse. It has learned that saying "disease" more often is rewarded, without learning who is
actually ill.

In Part 2 you will watch the same trick work spectacularly on the houses. **That
inconsistency is the finding.** In-context learning is cheap to try and genuinely useful; it
is not something you can assume will work, and it bills you for those examples on every
single prediction, forever, instead of once.

**4. Half of what looks like a model difference is a benchmark-design difference.**
The two trained-model rows are the *same model*, on the *same patients*, differing by one
number that nobody would think to report. Before concluding that approach A beats approach B,
check that you have not simply handed B a worse question.

**5. Look at the confidence interval before you believe any of it.**
On a few dozen patients a 95% interval spans something like ±15 percentage points — check the
column and see. Two approaches whose intervals overlap have **not** been distinguished by this
experiment, however confident the bar chart looks. If you take one habit
from this notebook, take that one: *a benchmark this size cannot settle a close call, and
most published comparisons are this size.*

### ✏️ Task 1 — The wording is the model (7 min)

Change **one thing** in the prompt below and re-run the benchmark on a smaller sample. Ideas,
roughly in order of how much they usually matter:

- Tell the model the **base rate**: *"About 10% of patients in this study have the disease."*
  (Ours is a balanced sample, so watch what that does to precision and recall.)
- Give the markers **meaning**: *"Blood marker A is an inflammation marker; values above 4
  are considered elevated."* You are injecting knowledge the model could not have.
- Ask it to **think first**: add *"Reason briefly, then give the number on the last line."*
  and raise `max_tokens` to 300. This usually helps — and multiplies the cost.
- Or make it **worse** on purpose: strip the units, or call the fields `f1`…`f6`.

Twelve patients is a *very* small sample — small enough that one lucky patient moves the score
by 8 points. Treat this as a way to see prompts working, not as evidence.

Re-run and compare against the row you already have. Did your prompt change the score by more
than the confidence interval?

In [18]:
# ✏️ TASK 1 — edit the prompt text, then run.
# Uses TASK_PATIENTS from the settings cell: a smaller sample again, so you can iterate fast.

def my_patient_risk(patient):
    prompt = (
        "You are a screening tool for a research study. This is synthetic teaching data.\n"
        # ✏️ ADD OR CHANGE A LINE HERE ------------------------------------------------
        "Estimate the probability that this patient has the disease.\n"
        # ------------------------------------------------------------------------------
        "Answer with a single number from 0 to 100 and nothing else.\n\n"
        f"{patient_to_text(patient)}"
    )
    percent = to_number(ask_llm(prompt, max_tokens=10))
    return None if percent is None else min(max(percent, 0.0), 100.0) / 100


if LLM_READY:
    sample = bench.patients.head(TASK_PATIENTS)
    my_risk = run_in_parallel(my_patient_risk, sample.itertuples())
    my_pred = [None if r is None else int(r >= 0.5) for r in my_risk]

    my_truth = sample["has_disease"].to_numpy()
    show_scoreboard({
        "Class 1's model": score_classification(
            my_truth, bench.model_pred_tuned[:TASK_PATIENTS],
            risk=bench.model_risk[:TASK_PATIENTS]),
        "My prompt": score_classification(my_truth, my_pred, risk=my_risk),
    }, title=f"My prompt, on the first {TASK_PATIENTS} patients")

············  (12 calls)
🏁 My prompt, on the first 12 patients



Approach,Accuracy,95% CI,Precision,Recall,F1,Caught,Missed,False alarms,AUC
Class 1's model,67%,39%–86%,67%,67%,0.67,4,2,2,0.778
My prompt,67%,39%–86%,75%,50%,0.60,3,3,1,0.708


## Part 2: Class 2's Benchmark — Ten Houses, Tested Honestly ⏱️ 20 min

### 🏠 A confession about Class 2

Class 2 fitted a line to ten houses and reported its error **on those same ten houses**. That
is the exam-with-the-answer-key problem Class 1 warned about, and we did it anyway because
ten points do not survive a train/test split.

Class 2's whole data set is ten houses, so there is nothing to sample here — we use all of
them, and it still costs only twenty calls.

There is a fix for the exam-with-the-answer-key problem, and it costs nothing:
**leave-one-out**. Hide one house. Fit the line on the
other nine. Predict the hidden one. Repeat ten times. Every prediction now comes from a line
that has never seen the house it is predicting — which is exactly the handicap the language
model has, and the only way this comparison means anything.

In [19]:
sizes, prices = np.array(HOUSE_SIZES), np.array(HOUSE_PRICES)

loo_line = leave_one_out_line(sizes, prices)

for size, price, guess in zip(sizes, prices, loo_line):
    print(f"   {size:,} sq ft → really ${price}k, line trained without it said ${guess:.0f}k")

   800 sq ft → really $150k, line trained without it said $142k
   1,000 sq ft → really $185k, line trained without it said $176k
   1,200 sq ft → really $215k, line trained without it said $211k
   1,400 sq ft → really $230k, line trained without it said $248k
   1,600 sq ft → really $270k, line trained without it said $280k
   1,800 sq ft → really $295k, line trained without it said $314k
   2,000 sq ft → really $350k, line trained without it said $345k
   2,200 sq ft → really $415k, line trained without it said $372k
   2,400 sq ft → really $410k, line trained without it said $414k
   2,600 sq ft → really $435k, line trained without it said $452k


### 🤖 The Same Two Conditions as Part 1

- **No examples.** *"A house is 1,800 square feet. What does it cost?"* The model has never
  seen our neighbourhood and does not know which city, year or currency we mean. It answers
  from whatever it absorbed about house prices while reading the internet.
- **The other nine sales pasted in.** Now it has **exactly the information the line has** —
  the same nine houses, the same prices, for the same held-out house. Two ways of using one
  set of facts.

That second condition is the fair fight, and it is the interesting one.

In [20]:
def price_no_examples(size):
    """Zero-shot: no data at all, just the question."""
    prompt = (f"A house is {size} square feet. Estimate its price in thousands of US dollars.\n"
              f"Answer with just a number, no words, no dollar sign. Example: 250")
    return to_price_in_thousands(ask_llm(prompt, max_tokens=20))


def price_with_examples(index):
    """In-context: the same nine sales the leave-one-out line was fitted on."""
    prompt = ("Here are recent house sales in one neighbourhood:\n"
              f"{other_houses_as_text(sizes, prices, index)}\n\n"
              f"Estimate the price of a {sizes[index]:,} sq ft house in the same neighbourhood, "
              f"in thousands of dollars.\n"
              f"Answer with just a number, no words, no dollar sign. Example: 250")
    return to_price_in_thousands(ask_llm(prompt, max_tokens=20))


llm_cold = llm_warm = None

if not LLM_READY:
    print("⏭️  This cell needs an API key — see the setup cell near the top.")
else:
    llm_cold = run_in_parallel(price_no_examples, sizes)
    llm_warm = run_in_parallel(price_with_examples, range(len(sizes)))

··········  (10 calls)
··········  (10 calls)


In [21]:
series = {"Class 2's line (leave-one-out)": loo_line}
if LLM_READY:
    series["LLM, no examples"] = llm_cold
    series[f"LLM, the other 9 sales"] = llm_warm

plot_price_comparison(sizes, prices, series)

errors = {name: score_regression(prices, values) for name, values in series.items()}
plot_mae_bars(errors)

for name, e in errors.items():
    note = f", {e['skipped']} reply/replies had no number" if e["skipped"] else ""
    print(f"{name:>32}:  average miss ${e['mae'] * 1000:,.0f}   "
          f"worst miss ${e['worst'] * 1000:,.0f}{note}")

  Class 2's line (leave-one-out):  average miss $13,620   worst miss $43,419
                LLM, no examples:  average miss $63,500   worst miss $165,000
          LLM, the other 9 sales:  average miss $14,400   worst miss $40,000


### 💬 Three Things Worth Arguing About

**1. The cold model's error is not really about houses.** It is about *which market it
assumed*. Ask it about an 1,800 sq ft house and it has to silently pick a country, a decade
and a neighbourhood before it can answer. If it happens to pick one like ours, it looks
excellent; if it picks San Francisco or rural Portugal, it looks absurd. **Neither outcome
tells you anything about its ability to do regression** — which is exactly why a benchmark
with a known answer is worth building.

**2. With the nine sales in the prompt, it usually gets close.** Sometimes very close. That is
worth sitting with: given identical information, "read nine numbers and infer the pattern"
is something these models can genuinely do, with no fitting step at all. If you have ever
needed a rough model in the next ten minutes rather than the next ten days, this is the
capability you were looking for.

**3. And the line still gives you something the model never will.** `$167 per square foot` is
a sentence you can take to a colleague, argue with, check against another neighbourhood, and
defend in front of a regulator. Ask the model *why* it said 350 and you get a fluent
paragraph that may have nothing to do with how it actually produced the number. On ten data
points the line also cost you approximately zero dollars and one millisecond.

> 📌 **If you have a table of numbers and a column to predict, fit the small model.** That
> advice survived this benchmark. What changed is the honest footnote: *and if you don't have
> the table yet, the language model will give you a usable answer this afternoon.*

### ✏️ Task 2 — Push both of them off the edge of the data (5 min)

Class 2 warned about **extrapolation**: our houses run from 800 to 2,600 sq ft, and the line
has no idea what happens outside that range — it just keeps going straight.

Ask both approaches about a house far outside the data. The line will answer with total
confidence. So will the model. **Only one of them is being confident about arithmetic.**

In [ ]:
# ✏️ TASK 2 — change the size, then run. Try 5000, then 200, then 50000.

BIG_HOUSE = 5000

from sklearn.linear_model import LinearRegression
full_line = LinearRegression().fit(sizes.reshape(-1, 1), prices)

print(f"Class 2's line says:            ${full_line.predict([[BIG_HOUSE]])[0]:,.0f}k")

if LLM_READY:
    prompt = ("Here are recent house sales in one neighbourhood:\n"
              + "\n".join(f"{s:,} sq ft sold for ${p}k" for s, p in zip(sizes, prices))
              + f"\n\nEstimate the price of a {BIG_HOUSE:,} sq ft house in the same "
                f"neighbourhood, in thousands of dollars.\n"
                f"Answer with just a number, no words, no dollar sign. Example: 250")
    print(f"The model, given all ten sales: ${to_price_in_thousands(ask_llm(prompt, max_tokens=20)):,.0f}k")

print("\n🤔 Which one would you put in a report — and could you defend either?")

## Part 3: The Two Columns Nobody Benchmarks ⏱️ 15 min

Accuracy is the column everyone reports. **Cost and latency are the columns that decide
whether the thing ever ships.** Let's measure both, since we just generated the data.

In [ ]:
# One call, timed honestly — sequentially, the way a live system would do it.
llm_seconds = float("nan")
if LLM_READY:
    start = time.time()
    ask_llm("Reply with the single word: ready.", max_tokens=5)
    llm_seconds = time.time() - start

# The trained model, on all 3,000 held-out patients at once.
start = time.time()
bench.model.predict(bench.X_test)
model_seconds = (time.time() - start) / len(bench.X_test)

print(f"⏱️  Language model:  {llm_seconds:.2f} seconds per prediction")
print(f"⏱️  Class 1's model: {model_seconds * 1e6:.1f} microseconds per prediction")
if llm_seconds == llm_seconds:      # not NaN
    print(f"\n    The trained model is about {llm_seconds / model_seconds:,.0f}× faster.")

In [ ]:
print_usage()
print()
per_call = cost_projection(USAGE, predictions_per_day=10_000, seconds_per_call=llm_seconds)

In [ ]:
if per_call:
    plot_cost_scaleup(per_call)

### 💬 Put That Number Somewhere You Will See It Again

The per-prediction cost looks like a rounding error, and at ten predictions a day it is. The
chart above is on a **logarithmic** axis for a reason.

Two things make it worse than it first looks:

- **This was already the cheap model.** Everything above ran on a small, fast one. A frontier
  model would multiply that bill by roughly ten for the same 130 calls — which is why "which
  is the smallest model that still passes my benchmark?" is a question worth an afternoon.
- **The prompt is charged every time.** Those example patients in Part 1 were re-sent with
  every single question — look at the input-token count above and divide. Class 1's model absorbed 7,000 patients once, during a training run
  that cost a second of laptop time, and has charged nothing since.
- **Nobody prices the retries, the failed parses, or the second call you add later** when you
  discover the first one needs the model to "think briefly first".

And the trade underneath all of it:

> **Your labelling time up front, or a per-prediction fee forever.** That is the actual
> decision behind most "should we use AI for this?" conversations, and it is a finance
> question at least as much as a modelling one.

## The Scoreboard, Assembled

| | **Train a small model** (Classes 1–2) | **Ask a language model** (today) |
|---|---|---|
| **Labelled data needed** | thousands of rows | none to start; a few dozen help a lot |
| **Time to a first version** | days (mostly labelling) | minutes |
| **On this notebook's tabular benchmarks** | ✅ won the patients clearly, edged the houses | ⚠️ well behind on the patients; close on the houses *once given the same nine sales* |
| **On free text with no labels** | ❌ not possible at all | ✅ won outright |
| **Cost per prediction** | ~free | a fraction of a cent on a *small* model, **forever** |
| **Speed per prediction** | microseconds | ~1–3 seconds |
| **Same answer every time?** | ✅ always | ⚠️ usually at temperature 0, never guaranteed |
| **Can you read the rule?** | ✅ `$167 per sq ft` | ❌ a fluent story, not the reason |
| **Change the task** | new labels, retrain | rewrite one sentence |
| **Unparseable output** | impossible | a real, recurring failure mode |

### 🧭 A Rule of Thumb You Can Actually Use

- **Numbers in a spreadsheet, and you have history?** → train the small model. It won every
  benchmark in this notebook, and it wins on cost and latency by four orders of magnitude.
- **Text, images, or a task nobody has labelled yet?** → the language model, without
  hesitation. Part 0 was not close.
- **Millions of predictions a day?** → the small model, on the bill alone.
- **Not sure the task is even worth doing?** → prompt it this afternoon. A one-hour
  feasibility test beats a three-week project that answers the same question. This is the
  most underrated use of these models, and it is the one this whole notebook is built on:
  *we found out where the model stands in about forty minutes and a dollar.*

## 🎓 Wrap-Up: Six Ideas Worth Keeping

1. **The same model can be the best and the worst tool in the room, twenty minutes apart.**
   It beat every baseline on sentiment and lost to three milliseconds of logistic regression
   on the patients. "Is AI good at this?" is not a question with an answer; **"good at *what*,
   measured *how*?"** is.

2. **World knowledge substitutes for training data — right up until your columns are made
   up.** `age` and `smoker` it knows. `marker_a` it cannot possibly know, and `marker_a` was
   where the signal was. Most real business tables look like `marker_a`.

3. **In-context learning is real, cheap to try, and not dependable.** Pasting solved examples
   into the prompt transformed the house predictions and did much less for the patients — and
   you only know which you got by measuring. It also gets re-billed on every prediction,
   forever, instead of once.

4. **You still need labelled data — to *trust* it, not to *train* it.** Every honest sentence
   in this notebook came from held-out data with known answers. Without that, we would have
   a demo and a feeling.

5. **Bigger is not the question; "big enough" is.** We ran the whole benchmark on a small,
   fast model, and the interesting comparison was never model-vs-model — it was
   model-vs-logistic-regression-vs-doing-nothing. Reach for a larger model when your own
   benchmark says the small one is not enough, not before.

6. **Look at the confidence interval before you believe the winner.** Fifty patients gives
   you roughly ±15 points, and we said so in a column rather than hiding it. Most benchmark
   comparisons you will read — including the ones in vendor slide decks — are decided by
   margins smaller than their own error bars.

### 🧭 Discussion Questions

1. We made the benchmark set 50/50 instead of 10/90 so that recall was measurable. Name two
   ways that decision could mislead someone reading only our accuracy column.
2. The language model was never told what `marker_a` measures. If you *invented* a plausible
   meaning for it in the prompt and the score went up, what exactly would you have learned —
   about the model, or about the data?
3. Your team must screen 50,000 patients a month. Argue both sides on cost, speed, accuracy,
   auditability and time-to-launch — then choose, and say what would change your mind.
4. Part 2's cold model guessed house prices with no idea which country it was in, and
   sometimes landed close. Is that a success? What would you have to measure to find out?

### ✅ Before You Use Any of This On Something Real

- [ ] Do I have held-out labelled examples to **measure** it, not a handful I eyeballed?
- [ ] Is my benchmark **big enough** that the winner is outside the error bars?
- [ ] Did I build the **boring baseline** — a small trained model, or a keyword rule — and
      does the expensive option beat it by enough to justify itself?
- [ ] Do I know the **cost and latency** at my real volume, including the prompt I re-send?
- [ ] Do I know **which cases** it fails on, and are they the ones that matter most?
- [ ] Is there a plan for **unparseable replies**, because there will be some?

### 📚 Where to Go Next

- [`05_agentic_ai_simple.ipynb`](05_agentic_ai_simple.ipynb) — the gentle version of this
  class, if you want the ideas without the benchmark machinery.
- [`05_agentic_ai.ipynb`](05_agentic_ai.ipynb) — the full Class 5: **RAG**, giving a model
  your *own* documents so it stops guessing, plus tool-using **agents** and prompt injection.
- [`self_learning/05_agentic_ai.ipynb`](self_learning/05_agentic_ai.ipynb) — a ~3-hour deep
  dive that builds a small language model from scratch.